# 🧠 Explainable Brain Tumor MRI Classification — Combined Pipeline
## v1 (Multi-Model Comparison) + v2 (EfficientNet-B3 High-Performance) + Interactive MRI Upload

---
**Dataset:** [Brain Tumor MRI Dataset — Masoud Nickparvar (Kaggle)](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset)  
**Classes:** Glioma · Meningioma · No Tumor · Pituitary  
**Framework:** PyTorch + timm + pytorch-grad-cam

### What this notebook does
| Section | Description |
|---------|-------------|
| §1–3    | Setup, dataset download, data loaders |
| §4–5    | CBAM attention + advanced loss functions |
| §6      | **v1 comparison models** — B0, ResNet18, MobileNetV2 (±CBAM) |
| §7      | **v2 primary model** — EfficientNet-B3 + CBAM + Mixup + Label Smoothing + TTA |
| §8      | Ablation study (4 CBAM × fine-tune configurations) |
| §9      | Grad-CAM explainability — all models side-by-side |
| §10     | Error analysis on misclassified images |
| §11     | **v1 vs v2 comparison table & bar chart** |
| §12     | 🆕 **Interactive MRI upload → prediction + Grad-CAM panel** |
| §13     | Save models + final summary |

> **Runtime:** GPU required (Runtime → Change runtime type → T4 GPU).  
> Full pipeline (all models) ≈ 3–4 hours on T4. Use `QUICK_MODE = True` for a fast 30-min sanity check.


---
## §1 — Environment Setup

In [ ]:
# Install all dependencies
!pip install -q timm grad-cam kaggle seaborn scikit-learn opencv-python-headless ipywidgets
print("✅ Packages installed")

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader, Subset, random_split

import timm
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_recall_fscore_support, f1_score
)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import numpy as np
import pandas as pd
import cv2, os, math, time, warnings, zipfile
from PIL import Image
from pathlib import Path
from collections import Counter
warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# ══════════════════════════════════════════════════
# ▶ QUICK_MODE — set True for fast 30-min test run
#   (2+5 epochs per model instead of full schedule)
QUICK_MODE = False
# ══════════════════════════════════════════════════

# Global constants
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_WORKERS = 2
NUM_CLASSES = 4
CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']
DISPLAY_NAMES = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
PALETTE = ['#E55A5A', '#5A8DE5', '#5AE57E', '#E5C85A']
MEAN    = [0.485, 0.456, 0.406]
STD     = [0.229, 0.224, 0.225]

print("\n✅ Imports & config ready")
print(f"   QUICK_MODE = {QUICK_MODE}")

---
## §2 — Dataset setup
> Download the [Brain Tumor MRI Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset) yourself. Set `BRAIN_TUMOR_DATASET_DIR` to its extracted root, or place it in a local `data/` directory. The dataset is not included in this repository.

In [ ]:
# ── Dataset location ──────────────────────────────────────────────────────────
DATA_ROOT = Path(os.environ.get('BRAIN_TUMOR_DATASET_DIR', 'data')).expanduser().resolve()
TRAIN_DIR = DATA_ROOT / 'Training'
TEST_DIR = DATA_ROOT / 'Testing'

if not TRAIN_DIR.is_dir() or not TEST_DIR.is_dir():
    raise FileNotFoundError(
        'Dataset not found. Set BRAIN_TUMOR_DATASET_DIR to the extracted dataset root '
        'containing Training/ and Testing/ folders.'
    )

print(f'Dataset root: {DATA_ROOT}')

In [ ]:
# Verify the expected class folders and display their image counts.
for split, d in [('Train', TRAIN_DIR), ('Test', TEST_DIR)]:
    if os.path.isdir(d):
        classes = sorted(os.listdir(d))
        for c in classes:
            n = len(os.listdir(os.path.join(d, c)))
            print(f"  {split}/{c}: {n}")
    else:
        print(f"  ⚠️  {d} not found!")

---
## §3 — Data Loaders, Augmentation & Visualization

In [ ]:
# ── Transforms ───────────────────────────────────────────────────────────────
# v2 stronger augmentation pipeline
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.15),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.08),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=5),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# TTA transforms — 5 augmented versions averaged at inference
tta_transforms = [
    transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Resize((IMG_SIZE+20, IMG_SIZE+20)), transforms.CenterCrop(IMG_SIZE), transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.RandomRotation((10, 10)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.RandomRotation((-10, -10)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
]

# ── Datasets ─────────────────────────────────────────────────────────────────
full_train = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
val_source = datasets.ImageFolder(TRAIN_DIR, transform=eval_transforms)
test_ds    = datasets.ImageFolder(TEST_DIR,  transform=eval_transforms)

val_size   = int(0.20 * len(full_train))
train_size = len(full_train) - val_size
train_index_split, val_index_split = random_split(
    range(len(full_train)), [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)
train_ds = Subset(full_train, train_index_split.indices)
val_ds = Subset(val_source, val_index_split.indices)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Classes : {full_train.classes}")
print(f"Train   : {train_size}  |  Val: {val_size}  |  Test: {len(test_ds)}")

In [ ]:
# ── Helper: denormalize for display ─────────────────────────────────────────
def denorm(tensor):
    t = tensor.clone()
    for c, m, s in zip(range(3), MEAN, STD):
        t[c] = t[c] * s + m
    return t.permute(1, 2, 0).clamp(0, 1).numpy()


# ── Class distribution ────────────────────────────────────────────────────────
train_counts = Counter(full_train.targets[i] for i in train_ds.indices)
test_counts  = Counter(lbl for _, lbl in test_ds.samples)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Dataset Class Distribution', fontsize=14, fontweight='bold')
for ax, (counts, title) in zip(axes, [
    (train_counts, 'Training Set'), (test_counts, 'Test Set')
]):
    vals = [counts[i] for i in range(NUM_CLASSES)]
    bars = ax.bar(DISPLAY_NAMES, vals, color=PALETTE, edgecolor='white', linewidth=0.8)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Count'); ax.set_ylim(0, max(vals) * 1.2)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8, str(v),
                ha='center', fontsize=10, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight'); plt.show()


# ── Sample grid: original vs augmented ───────────────────────────────────────
fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(14, 7))
fig.suptitle('Sample MRI Images — Original vs Augmented', fontsize=13, fontweight='bold')
for cls_idx, cls_name in enumerate(full_train.classes):
    idx = next(i for i, (_, lbl) in enumerate(full_train.samples) if lbl == cls_idx)
    full_train.transform = eval_transforms
    img_o, _ = full_train[idx]
    axes[0, cls_idx].imshow(denorm(img_o)); axes[0, cls_idx].set_title(f'{DISPLAY_NAMES[cls_idx]}\nOriginal', fontsize=9); axes[0, cls_idx].axis('off')
    full_train.transform = train_transforms
    img_a, _ = full_train[idx]
    axes[1, cls_idx].imshow(denorm(img_a)); axes[1, cls_idx].set_title('Augmented', fontsize=9); axes[1, cls_idx].axis('off')
full_train.transform = train_transforms   # restore
plt.tight_layout()
plt.savefig('sample_grid.png', dpi=150, bbox_inches='tight'); plt.show()

---
## §4 — Model Components
### 4a — CBAM Attention Module

In [ ]:
class ChannelAttention(nn.Module):
    """Which feature maps matter — learned via avg+max pooling through shared MLP."""
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        mid = max(in_channels // reduction, 8)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, in_channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        scale = self.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))
        return x * scale.unsqueeze(-1).unsqueeze(-1)


class SpatialAttention(nn.Module):
    """Where in the feature map to focus — 2D mask via conv(avg_ch ‖ max_ch)."""
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))


class CBAM(nn.Module):
    """Convolutional Block Attention Module (Woo et al., ECCV 2018)."""
    def __init__(self, in_channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel = ChannelAttention(in_channels, reduction)
        self.spatial = SpatialAttention(kernel_size)

    def forward(self, x):
        return self.spatial(self.channel(x))


# Sanity check
_x = torch.randn(2, 1280, 7, 7)
_c = CBAM(1280)
assert _c(_x).shape == _x.shape
del _x, _c
print("✅ CBAM defined and verified")

### 4b — Advanced Loss Functions (v2)

In [ ]:
class LabelSmoothingCE(nn.Module):
    """
    Label smoothing (ε=0.1): distributes 10% probability mass to wrong classes.
    Reduces overconfident predictions and closes the val→test generalisation gap.
    """
    def __init__(self, epsilon=0.1, num_classes=4):
        super().__init__()
        self.epsilon = epsilon; self.K = num_classes

    def forward(self, logits, targets):
        log_p = F.log_softmax(logits, dim=-1)
        nll   = -log_p.gather(-1, targets.unsqueeze(1)).squeeze(1)
        smooth= -log_p.mean(dim=-1)
        return ((1 - self.epsilon) * nll + self.epsilon * smooth).mean()


class FocalLoss(nn.Module):
    """
    Focal loss (γ=2, ε=0.1): down-weights easy examples, focuses on hard ones.
    Use if Glioma recall stays low after standard training.
    """
    def __init__(self, gamma=2.0, label_smooth=0.1):
        super().__init__()
        self.gamma = gamma; self.ls = label_smooth

    def forward(self, logits, targets):
        K   = logits.size(1)
        lp  = F.log_softmax(logits, dim=1)
        p   = lp.exp()
        with torch.no_grad():
            st = torch.full_like(logits, self.ls / K)
            st.scatter_(1, targets.unsqueeze(1), 1 - self.ls + self.ls / K)
        p_t = (p * st).sum(dim=1)
        return (-(1 - p_t) ** self.gamma * (lp * st).sum(dim=1)).mean()


print("✅ LabelSmoothingCE and FocalLoss defined")

### 4c — Mixup Augmentation (v2)

In [ ]:
def mixup_data(x, y, alpha=0.4):
    """
    Creates virtual training examples by linearly interpolating two images.
    Forces linear behaviour between classes → reduces Glioma/Meningioma confusion.
    """
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


def mixup_criterion(crit, logits, y_a, y_b, lam):
    return lam * crit(logits, y_a) + (1 - lam) * crit(logits, y_b)


print("✅ Mixup defined (alpha=0.4 recommended)")

---
## §5 — Model Definitions
### 5a — TumorClassifier (v1) — for comparison models

In [ ]:
FEAT_DIM = {
    'efficientnet_b0': 1280,
    'efficientnet_b3': 1536,
    'resnet18':         512,
    'mobilenetv2_100': 1280,
}


class TumorClassifier(nn.Module):
    """v1 model: backbone → CBAM → pool → dropout → linear. Used for comparison runs."""

    def __init__(self, backbone='efficientnet_b0', num_classes=4,
                 use_cbam=True, dropout=0.3):
        super().__init__()
        self.backbone_name = backbone
        self.use_cbam      = use_cbam
        self.features = timm.create_model(backbone, pretrained=True,
                                          num_classes=0, global_pool='')
        fdim = FEAT_DIM[backbone]
        self.cbam       = CBAM(fdim) if use_cbam else nn.Identity()
        self.pool       = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(fdim),
            nn.Dropout(dropout),
            nn.Linear(fdim, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.cbam(x)
        x = self.pool(x)
        return self.classifier(x)

    def freeze_backbone(self):
        for p in self.features.parameters(): p.requires_grad = False

    def unfreeze_last_blocks(self, n=2):
        for p in self.features.parameters(): p.requires_grad = True
        if 'efficientnet' in self.backbone_name:
            blocks = list(self.features.blocks.children())
            for blk in blocks[:-n]:
                for p in blk.parameters(): p.requires_grad = False
        # ResNet / MobileNet: unfreeze everything

    def param_count(self):
        tot = sum(p.numel() for p in self.parameters())
        tr  = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return tot, tr


print("✅ TumorClassifier (v1) defined")

### 5b — TumorClassifierV2 — EfficientNet-B3 high-performance model

In [ ]:
class TumorClassifierV2(nn.Module):
    """
    v2 model: EfficientNet-B3 + CBAM + richer head.
    B3 has 1536-dim features vs B0's 1280 — better discrimination for hard cases.
    Dropout 0.4 (was 0.3) + extra BN layer for stronger regularisation.
    """

    def __init__(self, backbone='efficientnet_b3', num_classes=4,
                 use_cbam=True, dropout=0.4):
        super().__init__()
        self.backbone_name = backbone
        self.use_cbam = use_cbam
        self.features = timm.create_model(backbone, pretrained=True,
                                          num_classes=0, global_pool='')
        fdim = FEAT_DIM[backbone]
        self.cbam       = CBAM(fdim) if use_cbam else nn.Identity()
        self.pool       = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(fdim),
            nn.Dropout(dropout),
            nn.Linear(fdim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout / 2),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.cbam(x)
        x = self.pool(x)
        return self.head(x)

    def freeze_backbone(self):
        for p in self.features.parameters(): p.requires_grad = False

    def unfreeze_last_blocks(self, n=3):
        """Unfreeze last n blocks + conv_head + bn2 (3 blocks for v2 vs 2 for v1)."""
        for p in self.features.parameters(): p.requires_grad = True
        if 'efficientnet' in self.backbone_name:
            keys = [f'blocks.{6-i}' for i in range(n)] + ['conv_head', 'bn2']
            for name, p in self.features.named_parameters():
                if not any(k in name for k in keys):
                    p.requires_grad = False
        elif 'resnet' in self.backbone_name:
            for name, p in self.features.named_parameters():
                if not any(f'layer{4-i}' in name for i in range(n)):
                    p.requires_grad = False

    def param_count(self):
        tot = sum(p.numel() for p in self.parameters())
        tr  = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return tot, tr


# Quick shape check
_m = TumorClassifierV2().to(DEVICE)
_o = _m(torch.randn(2, 3, 224, 224).to(DEVICE))
assert _o.shape == (2, 4), f"Bad output: {_o.shape}"
tot, tr = _m.param_count()
print(f"✅ TumorClassifierV2 verified | Total: {tot/1e6:.2f}M | Trainable: {tr/1e6:.2f}M")
del _m, _o

---
## §6 — Training & Evaluation Engine

In [ ]:
# ── Warmup + Cosine LR Scheduler ─────────────────────────────────────────────
class WarmupCosineScheduler:
    """Linear warmup then cosine annealing. Prevents unstable early gradients."""
    def __init__(self, optimizer, warmup_epochs, total_epochs, eta_min=1e-7):
        self.opt = optimizer
        self.warmup = warmup_epochs
        self.total  = total_epochs
        self.eta    = eta_min
        self.base_lrs = [g['lr'] for g in optimizer.param_groups]
        self.epoch = 0

    def step(self):
        e = self.epoch; self.epoch += 1
        if e < self.warmup:
            scale = (e + 1) / self.warmup
        else:
            prog  = (e - self.warmup) / max(1, self.total - self.warmup)
            scale = self.eta + 0.5 * (1 - self.eta) * (1 + math.cos(math.pi * prog))
        for g, base in zip(self.opt.param_groups, self.base_lrs):
            g['lr'] = base * scale


# ── Epoch-level functions ─────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion,
                use_mixup=False, mixup_alpha=0.4, clip_grad=1.0):
    model.train()
    tot_loss = correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        if use_mixup:
            imgs, ya, yb, lam = mixup_data(imgs, labels, mixup_alpha)
            logits = model(imgs)
            loss   = mixup_criterion(criterion, logits, ya, yb, lam)
            preds  = logits.argmax(1)
            correct += (lam*(preds==ya).float() + (1-lam)*(preds==yb).float()).sum().item()
        else:
            logits = model(imgs)
            loss   = criterion(logits, labels)
            correct += (logits.argmax(1) == labels).sum().item()
        loss.backward()
        if clip_grad > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
        optimizer.step()
        tot_loss += loss.item() * imgs.size(0)
        total    += imgs.size(0)
    return tot_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion=None):
    model.eval()
    crit = criterion or nn.CrossEntropyLoss()
    tot_loss = correct = total = 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out = model(imgs)
        tot_loss += crit(out, labels).item() * imgs.size(0)
        preds = out.argmax(1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return tot_loss/total, correct/total, np.array(all_preds), np.array(all_labels)


# ── TTA Evaluation ────────────────────────────────────────────────────────────
def evaluate_with_tta(model, test_dir, tta_tfms, classes, tag=''):
    """Average softmax probabilities over 5 augmented passes — free +1-2% boost."""
    model.eval()
    tta_loaders = [
        DataLoader(datasets.ImageFolder(test_dir, transform=t),
                   batch_size=32, shuffle=False, num_workers=2)
        for t in tta_tfms
    ]
    n = len(tta_loaders[0].dataset)
    all_probs  = torch.zeros(n, len(classes))
    true_labels = []

    with torch.no_grad():
        for aug_idx, loader in enumerate(tta_loaders):
            b = 0
            for imgs, labels in loader:
                probs = torch.softmax(model(imgs.to(DEVICE)), 1).cpu()
                all_probs[b:b+len(imgs)] += probs
                if aug_idx == 0: true_labels.extend(labels.numpy())
                b += len(imgs)

    preds  = (all_probs / len(tta_tfms)).argmax(1).numpy()
    labels = np.array(true_labels)
    acc    = accuracy_score(labels, preds)
    _, _, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    return acc, f1, preds, labels


# ── Plot helpers ──────────────────────────────────────────────────────────────
def plot_curves(history, tag, s1_end=5):
    epochs = range(1, len(history['train_loss'])+1)
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    fig.suptitle(f'{tag} — Training History', fontsize=12, fontweight='bold')
    for ax, (kt, kv, yl, title) in zip(axes, [
        ('train_loss','val_loss','Loss','Loss'),
        ('train_acc','val_acc','Accuracy (%)','Accuracy'),
        ('lr', None, 'LR','Learning Rate'),
    ]):
        sc = 100 if 'acc' in kt else 1
        ax.plot(epochs, [v*sc for v in history[kt]], 'b-o', ms=3, label='Train')
        if kv: ax.plot(epochs, [v*sc for v in history[kv]], 'r-s', ms=3, label='Val')
        ax.axvline(s1_end+0.5, color='gray', ls='--', alpha=0.5, label='Stage 1→2')
        ax.set_xlabel('Epoch'); ax.set_ylabel(yl); ax.set_title(title, fontweight='bold')
        ax.legend(); ax.grid(alpha=0.3); ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(f"{tag.replace('+','_').replace(' ','_')}_curves.png", dpi=150, bbox_inches='tight')
    plt.show()


def print_eval(preds, labels, tag):
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    print(f"\n{'═'*55}")
    print(f"  {tag}")
    print(f"  Accuracy  : {acc*100:.2f}%  |  F1 (weighted): {f1*100:.2f}%")
    print(f"{'═'*55}")
    print(classification_report(labels, preds, target_names=DISPLAY_NAMES, digits=4))
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(confusion_matrix(labels, preds), display_labels=DISPLAY_NAMES).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{tag}\nConfusion Matrix', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"cm_{tag.replace(' ','_').replace('+','').lower()}.png", dpi=150, bbox_inches='tight')
    plt.show()
    return {'accuracy': acc, 'precision': p, 'recall': r, 'f1': f1}


print("✅ Training engine, schedulers, eval functions ready")

---
## §7 — v1: Comparison Model Training
Trains B0+CBAM, B0 no-CBAM, ResNet18+CBAM, MobileNetV2+CBAM using the v1 architecture.  
Expected runtime: **~60–90 min** on T4 (full). Set `QUICK_MODE = True` for 2+5 epochs each.


In [ ]:
def train_v1_model(backbone, use_cbam, tag,
                   s1_ep=5, s2_ep=10, lr1=1e-3, lr2=1e-4):
    """Two-stage v1 training: frozen backbone → fine-tune last 2 blocks."""
    if QUICK_MODE: s1_ep, s2_ep = 2, 5

    print(f"\n{'='*55}")
    print(f"  [{tag}]  backbone={backbone}  CBAM={use_cbam}")
    print(f"  Stage1={s1_ep}ep  Stage2={s2_ep}ep")
    print(f"{'='*55}")

    model = TumorClassifier(backbone, use_cbam=use_cbam).to(DEVICE)
    crit  = nn.CrossEntropyLoss(label_smoothing=0.05)
    ecrit = nn.CrossEntropyLoss()
    history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[], 'lr':[]}
    best_va, best_state = 0.0, None

    # Stage 1 — frozen
    model.freeze_backbone()
    opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                             lr=lr1, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr1,
              steps_per_epoch=len(train_loader), epochs=s1_ep)
    for ep in range(1, s1_ep+1):
        tl, ta = train_epoch(model, train_loader, opt, crit, use_mixup=False)
        vl, va, _, _ = eval_epoch(model, val_loader, ecrit)
        sched.step()
        history['train_loss'].append(tl); history['val_loss'].append(vl)
        history['train_acc'].append(ta);  history['val_acc'].append(va)
        history['lr'].append(opt.param_groups[0]['lr'])
        print(f"  [S1] ep{ep:02d} loss={tl:.4f}/{vl:.4f} acc={ta*100:.1f}%/{va*100:.1f}%")
        if va > best_va: best_va = va; best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}

    # Stage 2 — unfreeze last 2 blocks
    model.unfreeze_last_blocks(2)
    opt2 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=lr2, weight_decay=1e-4)
    sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=s2_ep, eta_min=1e-6)
    for ep in range(1, s2_ep+1):
        tl, ta = train_epoch(model, train_loader, opt2, crit, use_mixup=False, clip_grad=1.0)
        vl, va, _, _ = eval_epoch(model, val_loader, ecrit)
        sched2.step()
        history['train_loss'].append(tl); history['val_loss'].append(vl)
        history['train_acc'].append(ta);  history['val_acc'].append(va)
        history['lr'].append(opt2.param_groups[0]['lr'])
        print(f"  [S2] ep{ep:02d} loss={tl:.4f}/{vl:.4f} acc={ta*100:.1f}%/{va*100:.1f}%")
        if va > best_va: best_va = va; best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}

    model.load_state_dict(best_state)
    print(f"  Best val acc: {best_va*100:.2f}%")
    return model, history, best_va


# ── Train all v1 comparison configs ──────────────────────────────────────────
V1_CONFIGS = [
    ('efficientnet_b0', True,  'EfficientNet-B0 + CBAM'),
    ('efficientnet_b0', False, 'EfficientNet-B0'),
    ('resnet18',        True,  'ResNet18 + CBAM'),
    ('mobilenetv2_100', True,  'MobileNetV2 + CBAM'),
]

v1_models   = {}
v1_histories= {}
v1_results  = {}

for backbone, use_cbam, tag in V1_CONFIGS:
    t0 = time.time()
    model, hist, best_va = train_v1_model(backbone, use_cbam, tag)
    _, _, preds, labels  = eval_epoch(model, test_loader)
    met = print_eval(preds, labels, tag)
    met['train_time'] = (time.time()-t0)/60
    met['params'] = model.param_count()[0] / 1e6
    v1_models[tag]    = model
    v1_histories[tag] = (hist, 5)  # (history, s1_end)
    v1_results[tag]   = met
    plot_curves(hist, tag, s1_end=5)
    torch.save(model.state_dict(), f"{tag.replace(' ','_').replace('+','').lower()}.pth")

print("\n✅ All v1 comparison models trained and saved")

---
## §8 — v2: EfficientNet-B3 + CBAM Primary Model
All upgrades combined: bigger backbone, label smoothing, Mixup, gradient clipping,  
warmup+cosine LR, 3-block unfreeze, TTA at inference.  
Expected runtime: **~90–120 min** full (30 epochs on T4). Set `QUICK_MODE = True` for 7 epochs.


In [ ]:
def train_v2_model(
    backbone='efficientnet_b3', use_cbam=True,
    s1_ep=5, s2_ep=25, lr1=3e-4, lr2=5e-5,
    use_mixup=True, mixup_alpha=0.4,
    loss_type='label_smooth', warmup_ep=3, tag=None
):
    """v2 full training: stronger backbone, mixup, label smoothing, warmup cosine LR."""
    if QUICK_MODE: s1_ep, s2_ep = 2, 5
    label = tag or (backbone + ('+CBAM' if use_cbam else ''))

    print(f"\n{'='*60}")
    print(f"  [{label}]")
    print(f"  Backbone={backbone}  CBAM={use_cbam}  Loss={loss_type}")
    print(f"  Stage1={s1_ep}ep lr={lr1}  Stage2={s2_ep}ep lr={lr2}  warmup={warmup_ep}ep")
    print(f"  Mixup={use_mixup} alpha={mixup_alpha}")
    print(f"{'='*60}")

    model = TumorClassifierV2(backbone, use_cbam=use_cbam).to(DEVICE)

    if loss_type == 'label_smooth': crit = LabelSmoothingCE(0.1, 4)
    elif loss_type == 'focal':      crit = FocalLoss(2.0, 0.1)
    else:                            crit = nn.CrossEntropyLoss()
    ecrit = nn.CrossEntropyLoss()

    history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[], 'lr':[]}
    best_va, best_state = 0.0, None

    # ── Stage 1: frozen backbone, no mixup (head needs clean signal) ──────────
    model.freeze_backbone()
    opt1 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=lr1, weight_decay=1e-4)
    for ep in range(1, s1_ep+1):
        t0 = time.time()
        tl, ta = train_epoch(model, train_loader, opt1, crit, use_mixup=False)
        vl, va, _, _ = eval_epoch(model, val_loader, ecrit)
        history['train_loss'].append(tl); history['val_loss'].append(vl)
        history['train_acc'].append(ta);  history['val_acc'].append(va)
        history['lr'].append(opt1.param_groups[0]['lr'])
        print(f"  [S1] ep{ep:02d}/{s1_ep} loss={tl:.4f}/{vl:.4f} "
              f"acc={ta*100:.1f}%/{va*100:.1f}% [{time.time()-t0:.0f}s]")
        if va > best_va: best_va=va; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}

    # ── Stage 2: unfreeze last 3 blocks, mixup + warmup cosine ───────────────
    model.unfreeze_last_blocks(n=3)
    tot, tr = model.param_count()
    print(f"  Stage2 start — Trainable: {tr/1e6:.2f}M / {tot/1e6:.2f}M")
    opt2 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=lr2, weight_decay=1e-4)
    sched = WarmupCosineScheduler(opt2, warmup_epochs=warmup_ep, total_epochs=s2_ep)

    for ep in range(1, s2_ep+1):
        t0 = time.time()
        tl, ta = train_epoch(model, train_loader, opt2, crit,
                             use_mixup=use_mixup, mixup_alpha=mixup_alpha)
        vl, va, _, _ = eval_epoch(model, val_loader, ecrit)
        sched.step()
        history['train_loss'].append(tl); history['val_loss'].append(vl)
        history['train_acc'].append(ta);  history['val_acc'].append(va)
        history['lr'].append(opt2.param_groups[0]['lr'])
        print(f"  [S2] ep{ep:02d}/{s2_ep} loss={tl:.4f}/{vl:.4f} "
              f"acc={ta*100:.1f}%/{va*100:.1f}% lr={opt2.param_groups[0]['lr']:.1e} [{time.time()-t0:.0f}s]")
        if va > best_va: best_va=va; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}

    model.load_state_dict(best_state)
    print(f"  Best val acc: {best_va*100:.2f}%")
    fname = label.replace('+','_').replace(' ','_') + '_v2.pth'
    torch.save(model.state_dict(), fname)
    print(f"  Saved: {fname}")
    return model, history


# ── Train primary model ───────────────────────────────────────────────────────
t_start = time.time()
model_b3, hist_b3 = train_v2_model(
    backbone='efficientnet_b3', use_cbam=True,
    s1_ep=5, s2_ep=25, lr1=3e-4, lr2=5e-5,
    use_mixup=True, mixup_alpha=0.4,
    loss_type='label_smooth', warmup_ep=3,
    tag='EfficientNet-B3+CBAM'
)
v2_train_time = (time.time() - t_start) / 60
plot_curves(hist_b3, 'EfficientNet-B3+CBAM', s1_end=5)

In [ ]:
# ── Evaluate v2 model (standard + TTA) ───────────────────────────────────────
_, _, std_preds, std_labels = eval_epoch(model_b3, test_loader)
std_metrics = print_eval(std_preds, std_labels, 'EfficientNet-B3+CBAM (standard)')

print("\n── Running TTA (5 augmented passes) ── approx 2-3 min ──")
tta_acc, tta_f1, tta_preds, tta_labels = evaluate_with_tta(
    model_b3, TEST_DIR, tta_transforms, DISPLAY_NAMES, tag='EfficientNet-B3+CBAM'
)
print(f"\n  TTA Accuracy : {tta_acc*100:.2f}%  |  F1: {tta_f1*100:.2f}%")
print(f"  TTA boost    : +{(tta_acc - std_metrics['accuracy'])*100:.2f}% over standard")

v2_results = {
    'EfficientNet-B3+CBAM (standard)': {
        'accuracy': std_metrics['accuracy'], 'f1': std_metrics['f1'],
        'precision': std_metrics['precision'], 'recall': std_metrics['recall'],
        'train_time': v2_train_time, 'params': model_b3.param_count()[0]/1e6
    },
    'EfficientNet-B3+CBAM (TTA)': {
        'accuracy': tta_acc, 'f1': tta_f1,
        'precision': None, 'recall': None,
        'train_time': None, 'params': None
    },
}
print("\n✅ v2 evaluation complete")

---
## §9 — Ablation Study
Four configurations on EfficientNet-B0 to isolate the contribution of CBAM and fine-tuning.


In [ ]:
ABLATION_CONFIGS = [
    ('A: No CBAM + Frozen only', False, 5, 0),
    ('B: No CBAM + Fine-tuned',  False, 2 if QUICK_MODE else 5, 3 if QUICK_MODE else 10),
    ('C: CBAM + Frozen only',    True,  5, 0),
    ('D: CBAM + Fine-tuned',     True,  2 if QUICK_MODE else 5, 3 if QUICK_MODE else 10),
]

ablation = {}
ecrit = nn.CrossEntropyLoss()

for cfg, use_cbam, s1, s2 in ABLATION_CONFIGS:
    print(f"\n── Ablation {cfg} ──")
    m = TumorClassifier('efficientnet_b0', use_cbam=use_cbam).to(DEVICE)
    crit_ = nn.CrossEntropyLoss()
    m.freeze_backbone()
    opt_ = torch.optim.AdamW(filter(lambda p: p.requires_grad, m.parameters()), lr=1e-3, weight_decay=1e-4)
    for _ in range(s1):
        train_epoch(m, train_loader, opt_, crit_, use_mixup=False)
    if s2 > 0:
        m.unfreeze_last_blocks(2)
        opt2_ = torch.optim.AdamW(filter(lambda p: p.requires_grad, m.parameters()), lr=1e-4, weight_decay=1e-4)
        sc = torch.optim.lr_scheduler.CosineAnnealingLR(opt2_, T_max=s2)
        for _ in range(s2):
            train_epoch(m, train_loader, opt2_, crit_, use_mixup=False)
            sc.step()
    _, _, p_, l_ = eval_epoch(m, test_loader, ecrit)
    acc_ = accuracy_score(l_, p_)
    _, _, f1_, _ = precision_recall_fscore_support(l_, p_, average='weighted')
    ablation[cfg] = {'accuracy': acc_*100, 'f1': f1_*100}
    print(f"  {cfg} → Acc={acc_*100:.2f}%  F1={f1_*100:.2f}%")

# ── Plot ablation ─────────────────────────────────────────────────────────────
cfgs   = list(ablation.keys())
colors = ['#e74c3c', '#3498db', '#f39c12', '#2ecc71']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Ablation Study — EfficientNet-B0', fontsize=13, fontweight='bold')
for ax, key, ttl in [(ax1,'accuracy','Accuracy (%)'), (ax2,'f1','Weighted F1 (%)')]:
    vals = [ablation[c][key] for c in cfgs]
    bars = ax.bar(range(len(cfgs)), vals, color=colors, edgecolor='black', linewidth=0.5)
    ax.bar_label(bars, fmt='%.2f%%', fontsize=10, fontweight='bold', padding=3)
    ax.set_xticks(range(len(cfgs))); ax.set_xticklabels(cfgs, rotation=12, ha='right')
    ax.set_title(ttl, fontweight='bold'); ax.set_ylim(max(0, min(vals)-8), 100)
    ax.grid(True, axis='y', alpha=0.3); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('ablation_study.png', dpi=150, bbox_inches='tight'); plt.show()

---
## §10 — Grad-CAM Explainability
For every model, each MRI shows: **Original | Heatmap | Overlay**  
Then a side-by-side panel compares With CBAM vs Without CBAM to show attention's effect.


In [ ]:
# ── Helper: get target layer per model ───────────────────────────────────────
def get_target_layer(model):
    """Returns the last convolutional layer appropriate for Grad-CAM."""
    if hasattr(model, 'features'):
        if hasattr(model.features, 'blocks'):
            return [model.features.blocks[-1]]
        # ResNet
        if hasattr(model.features, 'layer4'):
            return [model.features.layer4[-1]]
    return [list(model.modules())[-3]]   # fallback


# ── Per-model Grad-CAM grid: Original | Heatmap | Overlay ────────────────────
def show_gradcam_grid(model, test_dir, classes, n_per_class=2, title='Grad-CAM', save_path=None):
    target_layers = get_target_layer(model)
    cam = GradCAM(model=model, target_layers=target_layers)
    raw_tf = transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.ToTensor()])

    ncols = n_per_class * 3
    fig, axes = plt.subplots(len(classes), ncols, figsize=(ncols*3, len(classes)*3.2))
    fig.suptitle(title, fontsize=13, fontweight='bold')

    for cls_idx, cls_name in enumerate(classes):
        cls_dir = os.path.join(test_dir, cls_name if cls_name in os.listdir(test_dir) else cls_name.lower())
        fnames  = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.png','.jpeg'))][:n_per_class]
        for i, fname in enumerate(fnames):
            pil_img = Image.open(os.path.join(cls_dir, fname)).convert('RGB')
            raw  = raw_tf(pil_img).permute(1,2,0).numpy().astype(np.float32)
            inp  = eval_transforms(pil_img).unsqueeze(0).to(DEVICE)
            gcam = cam(inp, targets=[ClassifierOutputTarget(cls_idx)])[0]
            overlay = show_cam_on_image(raw, gcam, use_rgb=True, image_weight=0.5)
            heatmap = np.uint8(plt.cm.jet(gcam)[:,:,:3]*255)
            with torch.no_grad():
                logits = model(inp)
                pc = logits.argmax(1).item()
                cf = torch.softmax(logits,1)[0,pc].item()
            col = i*3
            axes[cls_idx,col].imshow((raw*255).astype(np.uint8))
            axes[cls_idx,col].set_ylabel(DISPLAY_NAMES[cls_idx], fontsize=9, fontweight='bold')
            axes[cls_idx,col].set_title('Original', fontsize=8); axes[cls_idx,col].axis('off')
            axes[cls_idx,col+1].imshow(heatmap); axes[cls_idx,col+1].set_title('Heatmap', fontsize=8); axes[cls_idx,col+1].axis('off')
            color = '#27ae60' if pc == cls_idx else '#e74c3c'
            axes[cls_idx,col+2].imshow(overlay)
            axes[cls_idx,col+2].set_title(f'Pred: {DISPLAY_NAMES[pc]}\n({cf*100:.1f}%)', fontsize=8, color=color)
            axes[cls_idx,col+2].axis('off')

    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    cam.__del__()


# Grad-CAM for all v1 models + v2 primary model
for tag, model in list(v1_models.items()) + [('EfficientNet-B3+CBAM (v2)', model_b3)]:
    show_gradcam_grid(model, TEST_DIR, full_train.classes, n_per_class=2,
                      title=f'Grad-CAM — {tag}',
                      save_path=f"gradcam_{tag.replace(' ','_').replace('+','').lower()}.png")

In [ ]:
# ── Side-by-side: With CBAM vs Without CBAM (B0) ────────────────────────────
model_with    = v1_models.get('EfficientNet-B0 + CBAM')
model_without = v1_models.get('EfficientNet-B0')

if model_with and model_without:
    n_imgs = 4
    fig, axes = plt.subplots(n_imgs, 3, figsize=(12, n_imgs*3.2))
    fig.suptitle('Grad-CAM Comparison — Effect of CBAM Attention', fontsize=13, fontweight='bold')
    axes[0,0].set_title('Original MRI',    fontsize=11, fontweight='bold')
    axes[0,1].set_title('Without CBAM',    fontsize=11, fontweight='bold', color='#e74c3c')
    axes[0,2].set_title('With CBAM',       fontsize=11, fontweight='bold', color='#27ae60')

    raw_tf = transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.ToTensor()])
    sample_idxs = np.random.choice(len(test_ds), n_imgs, replace=False)

    for row, idx in enumerate(sample_idxs):
        path, true_cls = test_ds.samples[idx]
        pil = Image.open(path).convert('RGB')
        raw = raw_tf(pil).permute(1,2,0).numpy().astype(np.float32)
        inp = eval_transforms(pil).unsqueeze(0).to(DEVICE)
        targets = [ClassifierOutputTarget(true_cls)]

        axes[row,0].imshow((raw*255).astype(np.uint8))
        axes[row,0].set_ylabel(DISPLAY_NAMES[true_cls], fontsize=9, fontweight='bold')
        axes[row,0].axis('off')

        for col, mdl in enumerate([model_without, model_with], start=1):
            tl = get_target_layer(mdl)
            c  = GradCAM(model=mdl, target_layers=tl)
            gs = c(inp, targets=targets)[0]
            ov = show_cam_on_image(raw, gs, use_rgb=True, image_weight=0.45)
            axes[row,col].imshow(ov); axes[row,col].axis('off')
            c.__del__()

    plt.tight_layout()
    plt.savefig('gradcam_cbam_comparison.png', dpi=150, bbox_inches='tight'); plt.show()

---
## §11 — Error Analysis
Shows the most confidently misclassified images for the best model (B3+CBAM).

In [ ]:
def error_analysis(model, dataset, loader, n_show=16, tag='Model', save_path=None):
    model.eval()
    all_probs, all_preds, all_labels, all_paths = [], [], [], []
    with torch.no_grad():
        for b_idx, (imgs, lbls) in enumerate(loader):
            logits = model(imgs.to(DEVICE))
            probs  = torch.softmax(logits, 1).cpu().numpy()
            preds  = np.argmax(probs, 1)
            start  = b_idx * loader.batch_size
            all_probs.extend(probs); all_preds.extend(preds); all_labels.extend(lbls.numpy())
            all_paths.extend([dataset.samples[i][0] for i in range(start, min(start+len(lbls), len(dataset.samples)))])

    mistakes = sorted([
        {'path': all_paths[i], 'true': all_labels[i], 'pred': all_preds[i], 'conf': all_probs[i][all_preds[i]]}
        for i in range(len(all_labels)) if all_labels[i] != all_preds[i]
    ], key=lambda x: x['conf'], reverse=True)

    print(f"\nMisclassified: {len(mistakes)} / {len(all_labels)} ({len(mistakes)/len(all_labels)*100:.1f}%)")
    patterns = Counter(f"{DISPLAY_NAMES[m['true']]} → {DISPLAY_NAMES[m['pred']]}" for m in mistakes)
    print("\nTop confusion patterns:")
    for pat, cnt in patterns.most_common(6): print(f"  {pat}: {cnt}")

    n   = min(n_show, len(mistakes))
    nc  = 4; nr = (n+nc-1)//nc
    fig, axes = plt.subplots(nr, nc, figsize=(nc*3.5, nr*3.5))
    fig.suptitle(f'Error Analysis — {tag}\n(Most Confidently Wrong)', fontsize=12, fontweight='bold')
    for i, err in enumerate(mistakes[:n]):
        img = cv2.cvtColor(cv2.resize(cv2.imread(err['path']), (IMG_SIZE,IMG_SIZE)), cv2.COLOR_BGR2RGB)
        ax  = axes.flat[i]
        ax.imshow(img)
        ax.set_title(f"True: {DISPLAY_NAMES[err['true']]}\nPred: {DISPLAY_NAMES[err['pred']]} ({err['conf']*100:.1f}%)",
                     fontsize=8, color='#e74c3c', fontweight='bold')
        ax.axis('off')
    for i in range(n, nr*nc): axes.flat[i].axis('off')
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    return mistakes


mistakes = error_analysis(model_b3, test_ds, test_loader, n_show=16,
                          tag='EfficientNet-B3+CBAM', save_path='error_analysis.png')

---
## §12 — v1 vs v2 Full Comparison Table

In [ ]:
# ── Build unified results table ───────────────────────────────────────────────
rows = []
for tag, res in v1_results.items():
    rows.append({
        'Model'            : f'{tag} (v1)',
        'Accuracy (%)'     : round(res['accuracy']*100, 2),
        'F1 (weighted, %)' : round(res['f1']*100, 2),
        'Params (M)'       : round(res.get('params', 0), 2),
        'Train Time (min)' : round(res.get('train_time', 0), 1),
        'TTA'              : 'No',
    })

for tag, res in v2_results.items():
    rows.append({
        'Model'            : tag,
        'Accuracy (%)'     : round(res['accuracy']*100, 2),
        'F1 (weighted, %)' : round(res['f1']*100, 2),
        'Params (M)'       : round(res.get('params') or 0, 2),
        'Train Time (min)' : round(res.get('train_time') or 0, 1),
        'TTA'              : 'Yes' if 'TTA' in tag else 'No',
    })

df = pd.DataFrame(rows).sort_values('Accuracy (%)', ascending=False).reset_index(drop=True)
print("\n" + "="*75)
print("  FULL v1 vs v2 COMPARISON")
print("="*75)
print(df.to_string(index=False))
print("="*75)
df.to_csv('model_comparison.csv', index=False)

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
bar_colors = ['#aaaaaa']*len(v1_results) + ['#1B6CA8','#00C6AE']
bars = ax.barh(df['Model'], df['Accuracy (%)'], color=bar_colors, edgecolor='white', linewidth=0.5)
ax.bar_label(bars, fmt='%.2f%%', padding=4, fontsize=9, fontweight='bold')
ax.axvline(90, color='red', linestyle='--', alpha=0.6, label='90% target')
ax.set_xlabel('Test Accuracy (%)', fontsize=11)
ax.set_title('v1 vs v2 — Test Accuracy Comparison', fontsize=13, fontweight='bold')
ax.set_xlim(60, 102); ax.legend()
ax.grid(True, axis='x', alpha=0.3); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('v1_vs_v2_comparison.png', dpi=150, bbox_inches='tight'); plt.show()

---
## §13 — 🆕 Interactive MRI Upload → Prediction + Grad-CAM Panel
Upload any MRI image and get:
- **Predicted class** with confidence bar chart
- **4-panel visual**: Original | Grad-CAM Heatmap | Overlay | Confidence bars


In [ ]:
# ── Build the interactive prediction function ─────────────────────────────────
def predict_and_explain(
    image_source,          # str path  OR  PIL Image  OR  numpy array (H,W,3 uint8)
    model=None,
    class_names=None,
    save_path='mri_prediction_panel.png'
):
    """
    Full pipeline: preprocess → predict → Grad-CAM → 4-panel display.

    Parameters
    ----------
    image_source : file path, PIL Image, or numpy RGB uint8 array
    model        : trained model (defaults to model_b3 — B3+CBAM)
    class_names  : list of class labels (defaults to DISPLAY_NAMES)
    save_path    : where to save the output panel image
    """
    if model is None:       model = model_b3
    if class_names is None: class_names = DISPLAY_NAMES
    model.eval()

    # ── Load & prepare image ──────────────────────────────────────────────────
    if isinstance(image_source, str):
        pil_img = Image.open(image_source).convert('RGB')
    elif isinstance(image_source, Image.Image):
        pil_img = image_source.convert('RGB')
    else:
        pil_img = Image.fromarray(image_source.astype(np.uint8)).convert('RGB')

    pil_resized = pil_img.resize((IMG_SIZE, IMG_SIZE))
    raw_np = np.array(pil_resized).astype(np.float32) / 255.0   # (H,W,3) float [0,1]

    inp_tensor = eval_transforms(pil_resized).unsqueeze(0).to(DEVICE)

    # ── Inference ─────────────────────────────────────────────────────────────
    with torch.no_grad():
        logits = model(inp_tensor)
        probs  = torch.softmax(logits, 1)[0].cpu().numpy()
        pred   = int(np.argmax(probs))
        conf   = float(probs[pred])

    # ── Grad-CAM ──────────────────────────────────────────────────────────────
    target_layers = get_target_layer(model)
    cam_obj       = GradCAM(model=model, target_layers=target_layers)
    targets       = [ClassifierOutputTarget(pred)]
    grayscale_cam = cam_obj(input_tensor=inp_tensor, targets=targets)[0]
    heatmap_rgb   = np.uint8(plt.cm.jet(grayscale_cam)[:, :, :3] * 255)
    overlay       = show_cam_on_image(raw_np, grayscale_cam, use_rgb=True, image_weight=0.5)
    cam_obj.__del__()

    # ── 4-panel figure ────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(18, 5))
    gs  = gridspec.GridSpec(1, 5, width_ratios=[1.1, 1, 1, 1, 1.4], wspace=0.05)

    # Title block
    ax0 = fig.add_subplot(gs[0])
    ax0.axis('off')
    correct_color = '#27ae60'
    ax0.text(0.5, 0.65, '🧠 MRI Prediction', ha='center', va='center',
             fontsize=13, fontweight='bold', transform=ax0.transAxes)
    ax0.text(0.5, 0.48, f'Predicted Class:', ha='center', va='center',
             fontsize=10, transform=ax0.transAxes, color='gray')
    ax0.text(0.5, 0.34, class_names[pred], ha='center', va='center',
             fontsize=16, fontweight='bold', color=correct_color, transform=ax0.transAxes)
    ax0.text(0.5, 0.20, f'Confidence: {conf*100:.1f}%', ha='center', va='center',
             fontsize=12, transform=ax0.transAxes, color='#2c3e50')
    ax0.set_facecolor('#f8f9fa')

    # Panel 1 — original
    ax1 = fig.add_subplot(gs[1])
    ax1.imshow((raw_np*255).astype(np.uint8))
    ax1.set_title('Original MRI', fontsize=10, fontweight='bold', pad=6)
    ax1.axis('off')

    # Panel 2 — heatmap
    ax2 = fig.add_subplot(gs[2])
    im = ax2.imshow(grayscale_cam, cmap='jet', vmin=0, vmax=1)
    ax2.set_title('Grad-CAM Heatmap', fontsize=10, fontweight='bold', pad=6)
    ax2.axis('off')
    plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)

    # Panel 3 — overlay
    ax3 = fig.add_subplot(gs[3])
    ax3.imshow(overlay)
    ax3.set_title('Heatmap Overlay', fontsize=10, fontweight='bold', pad=6)
    ax3.axis('off')

    # Panel 4 — probability bars
    ax4 = fig.add_subplot(gs[4])
    bar_cols = ['#27ae60' if i == pred else '#bdc3c7' for i in range(len(class_names))]
    bars     = ax4.barh(class_names, probs * 100, color=bar_cols,
                        edgecolor='white', linewidth=0.5, height=0.55)
    ax4.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9, fontweight='bold')
    ax4.set_xlabel('Probability (%)', fontsize=9)
    ax4.set_title('Class Probabilities', fontsize=10, fontweight='bold', pad=6)
    ax4.set_xlim(0, 115)
    ax4.spines[['top','right']].set_visible(False)
    ax4.grid(True, axis='x', alpha=0.3)

    fig.suptitle(
        f'EfficientNet-B3 + CBAM  |  Prediction: {class_names[pred]}  ({conf*100:.1f}% confidence)',
        fontsize=12, fontweight='bold', y=1.02, color=correct_color
    )

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n  Panel saved → {save_path}")
    return {'class': class_names[pred], 'confidence': conf, 'all_probs': probs}


print("✅ predict_and_explain() ready")
print("   Usage: predict_and_explain('your_mri.jpg')")
print("   or   : predict_and_explain(pil_image_object)")

In [ ]:
# ── Run the interactive upload widget ────────────────────────────────────────
from google.colab import files as colab_files

print("📤 Upload an MRI image file (.jpg / .png / .jpeg):")
print("   Tip: grab any image from the Testing/ folder or use your own scan\n")

uploaded_mri = colab_files.upload()   # opens file picker

if uploaded_mri:
    fname = list(uploaded_mri.keys())[0]
    print(f"\n✅ Received: {fname}")
    result = predict_and_explain(fname, model=model_b3, save_path='user_mri_prediction.png')
    print(f"\n  Predicted : {result['class']}")
    print(f"  Confidence: {result['confidence']*100:.1f}%")
    print("\n  Per-class breakdown:")
    for cls, p in zip(DISPLAY_NAMES, result['all_probs']):
        bar = '█' * int(p * 30)
        print(f"    {cls:15s}  {p*100:5.1f}%  {bar}")
else:
    print("No file uploaded — running on a random test image instead:")
    sample_path = test_ds.samples[np.random.randint(len(test_ds))][0]
    print(f"  Using: {sample_path}")
    predict_and_explain(sample_path, model=model_b3)

In [ ]:
# ── Batch demo: one image per class from the test set ────────────────────────
print("Running prediction on one example per class for demonstration:\n")
for cls_idx, cls_name in enumerate(full_train.classes):
    sample = next(path for path, lbl in test_ds.samples if lbl == cls_idx)
    print(f"── {DISPLAY_NAMES[cls_idx]} ──")
    predict_and_explain(sample, model=model_b3,
                        save_path=f'demo_{cls_name}.png')

---
## §14 — Save All Models & Final Summary

In [ ]:
import os

SAVE_DIR = Path(os.environ.get('BRAIN_TUMOR_OUTPUT_DIR', 'artifacts/models')).expanduser().resolve()

# create folder if it doesn't exist
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ── Save best v2 model ────────────────────────────────────
BEST_CKPT = SAVE_DIR / 'brain_tumor_B3_CBAM_best.pth'

torch.save({
    'model_state_dict' : model_b3.state_dict(),
    'backbone_name'    : model_b3.backbone_name,
    'use_cbam'         : model_b3.use_cbam,
    'num_classes'      : NUM_CLASSES,
    'class_names'      : DISPLAY_NAMES,
    'std_accuracy'     : std_metrics['accuracy'],
    'tta_accuracy'     : tta_acc,
    'macro_f1'         : std_metrics['f1'],
}, BEST_CKPT)

# ── Save all v1 models ────────────────────────────────────
for tag, mdl in v1_models.items():
    fname = f"v1_{tag.replace(' ','_').replace('+','').lower()}.pth"
    full_path = SAVE_DIR / fname
    torch.save(mdl.state_dict(), full_path)

print("✅ All models saved in:", SAVE_DIR)

In [ ]:
# ── Load best model helper (for future inference) ────────────────────────────
def load_best_model(ckpt_path=BEST_CKPT):
    ckpt  = torch.load(ckpt_path, map_location=DEVICE)
    model = TumorClassifierV2(
        backbone    = ckpt['backbone_name'],
        num_classes = ckpt['num_classes'],
        use_cbam    = ckpt['use_cbam'],
    )
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(DEVICE).eval()
    print(f"✅ Loaded: {ckpt['backbone_name']} | CBAM={ckpt['use_cbam']}")
    print(f"   Std accuracy : {ckpt.get('std_accuracy',0)*100:.2f}%")
    print(f"   TTA accuracy : {ckpt.get('tta_accuracy',0)*100:.2f}%")
    return model


# Verify round-trip
reloaded = load_best_model(BEST_CKPT)
_test_inp = torch.randn(1, 3, 224, 224).to(DEVICE)
with torch.no_grad(): _ = reloaded(_test_inp)
print("   Inference verified on reloaded model ✅")

In [ ]:
# ── Mount Google Drive and copy checkpoint (optional) ────────────────────────
# Uncomment to persist your model across Colab sessions:
#
# from google.colab import drive
# drive.mount('/content/drive')
# !cp brain_tumor_B3_CBAM_best.pth /content/drive/MyDrive/
# print("Copied to Google Drive")

# ── Final summary ─────────────────────────────────────────────────────────────
print("=" * 68)
print("  FINAL PROJECT RESULTS SUMMARY")
print("  Brain Tumor MRI Classification — Combined v1 + v2 Pipeline")
print("=" * 68)
print(f"\n  Primary model : EfficientNet-B3 + CBAM")
print(f"  Standard acc  : {std_metrics['accuracy']*100:.2f}%  |  F1: {std_metrics['f1']*100:.2f}%")
print(f"  TTA accuracy  : {tta_acc*100:.2f}%")
print(f"  TTA boost     : +{(tta_acc-std_metrics['accuracy'])*100:.2f}%")
print(f"  Train time    : {v2_train_time:.1f} min")
print()
print("  v1 Comparison models:")
print(f"  {'Model':<30} {'Accuracy':>10} {'F1':>8}")
print("  " + "─"*50)
for tag, res in v1_results.items():
    print(f"  {tag:<30} {res['accuracy']*100:>9.2f}% {res['f1']*100:>7.2f}%")
print()
print("  Ablation Study (EfficientNet-B0):")
print(f"  {'Config':<35} {'Accuracy':>10} {'F1':>8}")
print("  " + "─"*55)
for cfg, res in ablation.items():
    print(f"  {cfg:<35} {res['accuracy']:>9.2f}% {res['f1']:>7.2f}%")
print()
print("  v2 Upgrades vs v1 baseline:")
print("  [1] EfficientNet-B3 backbone   — 1536-dim vs 1280-dim features")
print("  [2] Label Smoothing ε=0.1      — closes val→test generalisation gap")
print("  [3] Mixup α=0.4                — softens Glioma/Meningioma boundary")
print("  [4] 30 total epochs (was 15)   — more training headroom")
print("  [5] Warmup + CosineAnnealingLR — stable Stage-2 fine-tuning")
print("  [6] Unfreeze 3 blocks (was 2)  — more backbone adaptation")
print("  [7] TTA 5 passes               — free +1-2% at inference")
print("  [8] Gradient clipping 1.0      — prevents exploding gradients")
print()
print("  Saved artifacts:")
artifacts = [
    BEST_CKPT,
    'model_comparison.csv',
    'v1_vs_v2_comparison.png',
    'ablation_study.png',
    'gradcam_cbam_comparison.png',
    'error_analysis.png',
    'class_distribution.png',
    'user_mri_prediction.png',
]
for a in artifacts:
    print(f"  {'✅' if os.path.exists(a) else '❌'} {a}")
print("=" * 68)